# Install dependencies

In [23]:
!pip install groq pypdf scikit-learn -q

# Set up Groq API key

In [24]:
import os
os.environ["GROQ_API_KEY"] = ""

# Upload PDF

In [25]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_name}")

Saving Atharva P. Matale - Resume.pdf to Atharva P. Matale - Resume (3).pdf
Uploaded: Atharva P. Matale - Resume (3).pdf


# Extract text and chunk it

In [26]:
from pypdf import PdfReader

def load_and_chunk(pdf_path, chunk_size=500):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks

chunks = load_and_chunk(pdf_name)
print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk:\n", chunks[0])

Total chunks created: 6

Sample chunk:
 ATHARVA P. MATALE
atharvamatale2907@gmail.com · github.com/oasis-parzival · linkedin.com/in/atharvamatale
EDUCATION
SIES Graduate School of Technology
Bachelor of Engineering — Computer Engineering
Navi Mumbai, India
TECHNICAL SKILLS
Core Foundations Deep Learning, Machine Learning, Computer Vision, Algorithms, Applied Mathematics, Discrete
Mathematics, Databases
Frameworks & Libraries PyTorch, MediaPipe, Hugging Face (Transformers, Accelerate), ONNX Runtime, OpenCV, Scikit-learn,
NumPy, Pandas



# Build the retriever (TF-IDF)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunks)

def retrieve_top_chunks(question, top_k=3):
    q_vector = vectorizer.transform([question])
    scores = cosine_similarity(q_vector, chunk_vectors)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    return [chunks[i] for i in top_indices]

# quick test
test_results = retrieve_top_chunks("What are the important topics?")
print("Top matching chunk:\n", test_results[0])

Top matching chunk:
 Optimization & Deployment Model Quantization (GGUF, AWQ), Layer Pruning, Context-Window Management, Edge Inference
Execution, Token Density Optimization
Languages & Architecture Python, C++, JavaScript (ES6+), SQL, Vector Databases (ChromaDB / Pinecone), WebSockets, REST APIs
EXPERIENCE
Zenher Mumbai, India
Founding Software Engineer — Machine Learning July 2025 – May 2026
Architected an end-to-end predictive menstruation ML pipeline and custom SLM framework from the ground up, boosting model
in


# Connect to Groq LLM

In [28]:
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def ask_question(question):
    relevant_chunks = retrieve_top_chunks(question)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""Answer the question using only the context below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}
"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test the full backend

In [30]:
print(ask_question("What are the projects?"))

SignBridge — Built an Indian Sign Language translation engine optimized with ONNX and native JavaScript, compressing model footprint by 40% to achieve client-side, webcam-based translation with near-zero latency (<10 ms).
BicepCurls-IDE — Developed a web-based, multi-agent AI IDE featuring automated code generation, real-time context-aware debugging, and self-correcting execution runtimes that cut software prototyping loops by 45%.


# Simple terminal chat loop

In [31]:
while True:
    q = input("\nAsk a question (or type 'exit'): ")
    if q.lower() == "exit":
        break
    print("\nAnswer:", ask_question(q))


Ask a question (or type 'exit'): what is the education

Answer: Bachelor of Engineering - Computer Engineering from SIES Graduate School of Technology.

Ask a question (or type 'exit'): exit


## **DAY 2 — Frontend (Gradio, custom styled)**

In [33]:
!pip install gradio -q

# Custom styling

In [42]:

custom_css = """
.gradio-container {
    max-width: 100% !important;
    padding: 0 !important;
}

#header {
    text-align: center;
    padding: 20px 12px 4px 12px;
}

#title-row {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 10px;
}

#title-icon {
    font-size: 1.8rem;
    line-height: 1;
}

#title {
    font-size: 1.8rem;
    font-weight: 700;
    background: linear-gradient(90deg, #6366f1, #ec4899);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin: 0;
}

#subtitle {
    color: #9ca3af;
    font-size: 0.9rem;
    margin-top: 6px;
}

#upload-accordion {
    max-width: 700px;
    margin: 12px auto 4px auto;
    padding: 0 12px;
}

#chat-wrap {
    max-width: 700px;
    margin: 8px auto 0 auto;
    padding: 0 12px;
}

#chatbot {
    height: 55vh !important;
    min-height: 350px;
    border-radius: 14px !important;
}

#input-row {
    max-width: 700px;
    margin: 8px auto 20px auto;
    padding: 0 12px;
}

footer {visibility: hidden}

@media (max-width: 640px) {
    #title, #title-icon { font-size: 1.4rem; }
    #subtitle { font-size: 0.8rem; }
    #chatbot { height: 50vh !important; }
}
"""

# Wrap existing functions for Gradio

In [37]:
def process_pdf(pdf_file):
    global chunks, vectorizer, chunk_vectors
    if pdf_file is None:
        return "⚠️ Please upload a PDF."
    chunks = load_and_chunk(pdf_file.name)
    vectorizer = TfidfVectorizer()
    chunk_vectors = vectorizer.fit_transform(chunks)
    return f"✅ Document indexed — {len(chunks)} chunks ready. Ask away!"

def ask_question_ui(question, history):
    if history is None:
        history = []
    if not chunks:
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": "⚠️ Please upload a PDF first."})
        return history

    answer = ask_question(question)  # reuses your Day 1 function as-is
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    return history

In [43]:

import gradio as gr

with gr.Blocks() as demo:
    gr.HTML("""
        <div id='header'>
            <div id='title-row'>
                <span id='title-icon'>📚</span>
                <span id='title'>Study Buddy</span>
            </div>
            <div id='subtitle'>Your AI-powered notes, syllabus & textbook assistant</div>
        </div>
    """)

    with gr.Column(elem_id="upload-accordion"):
        with gr.Accordion("📄 Upload a PDF to start chatting", open=True) as upload_section:
            pdf_input = gr.File(file_types=[".pdf"], height=90, show_label=False)
            status = gr.Textbox(show_label=False, interactive=False, container=False)

    with gr.Column(elem_id="chat-wrap"):
        chatbot = gr.Chatbot(
            show_label=False,
            elem_id="chatbot",
            avatar_images=(None, None),
            placeholder="Your conversation will appear here once you upload a PDF."
        )

    with gr.Row(elem_id="input-row"):
        question_box = gr.Textbox(
            show_label=False,
            placeholder="Ask a question about your document...",
            scale=8,
            container=False
        )
        ask_btn = gr.Button("➤", variant="primary", scale=1, min_width=48)

    def process_pdf_and_collapse(pdf_file):
        msg = process_pdf(pdf_file)
        return msg, gr.Accordion(open=False)  # auto-collapse once indexed

    pdf_input.change(
        fn=process_pdf_and_collapse,
        inputs=pdf_input,
        outputs=[status, upload_section]
    )
    ask_btn.click(fn=ask_question_ui, inputs=[question_box, chatbot], outputs=chatbot).then(
        lambda: "", None, question_box
    )
    question_box.submit(fn=ask_question_ui, inputs=[question_box, chatbot], outputs=chatbot).then(
        lambda: "", None, question_box
    )

demo.launch(share=True, css=custom_css, theme=gr.themes.Soft(primary_hue="indigo"))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://47a3a1bbd69fbc170f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
